# Mini notebook: robustez PACO (CPU vs GPU)

Valida que la aceleracion en GPU mantenga resultados cientificos comparables usando el mismo dataset y los mismos pixeles (`phi0s`).

In [ ]:
# Configuracion
DATASET_ROOT = "/content/drive/MyDrive/subchallenge1"  # o '/content/repos/tesis/subchallenge1'
INSTRUMENT = "lmircam"
DATASET_ID = 1

N_PIXELS = 2500
INNER_RADIUS = 8
OUTER_RADIUS = 55
USE_SUBPIXEL = False
CROP_HALF_SIZE = 100  # recorte a 200x200 para tiempos manejables

TOPK = 20
THRESHOLDS = (3.0, 5.0, 7.0)


In [ ]:
# Setup Colab
import os
import sys
import time
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
from astropy.io import fits

try:
    import cupy as cp  # noqa: F401
except Exception:
    print("Instalando CuPy...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "cupy-cuda12x"])

REPO_DIR = Path("/content/repos/tesis")
VIP_DIR = REPO_DIR / "VIP-master" / "VIP-master"

if not REPO_DIR.exists():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(["git", "clone", "https://github.com/Waofin/tesis", str(REPO_DIR)])

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(VIP_DIR)])
sys.path.insert(0, str(VIP_DIR / "src"))

import vip_hci as vip
from vip_hci.invprob.paco import FastPACO
from vip_hci.invprob import paco as paco_module

print("VIP:", vip.__version__)
print("Repo:", REPO_DIR)


In [ ]:
def load_dataset(base_dir, instrument, dataset_id):
    base = Path(base_dir)
    cube = fits.getdata(base / f"{instrument}_cube_{dataset_id}.fits")
    angles = fits.getdata(base / f"{instrument}_pa_{dataset_id}.fits").flatten()
    psf = fits.getdata(base / f"{instrument}_psf_{dataset_id}.fits")
    px = float(fits.getdata(base / f"{instrument}_pxscale_{dataset_id}.fits").flatten()[0])
    return cube, angles, psf, px


def crop_cube_center(cube, half_size=100):
    if cube.shape[1] <= 2 * half_size or cube.shape[2] <= 2 * half_size:
        return cube
    cy, cx = cube.shape[1] // 2, cube.shape[2] // 2
    return cube[:, cy-half_size:cy+half_size, cx-half_size:cx+half_size]


def build_phi0_annulus(img_shape, n_pixels=2500, inner=8, outer=55):
    h, w = int(img_shape[0]), int(img_shape[1])
    cy, cx = h // 2, w // 2
    outer_eff = min(outer, min(cy, cx) - 5)

    yy, xx = np.indices((h, w))
    rr = np.sqrt((yy - cy) ** 2 + (xx - cx) ** 2)
    mask = (rr >= inner) & (rr <= outer_eff)
    coords = np.column_stack((xx[mask], yy[mask]))

    if len(coords) == 0:
        raise RuntimeError("No se pudieron generar phi0s en el anillo")

    n_sel = min(n_pixels, len(coords))
    idx = np.linspace(0, len(coords) - 1, n_sel, dtype=int)
    return coords[idx].astype(np.int32)


def run_paco(cube, angles, psf, pixscale, phi0s, enable_gpu, use_subpixel=False):
    paco_module.enable_paco_gpu_backend(bool(enable_gpu))
    fwhm_arcsec = 4.0 * pixscale
    model = FastPACO(cube=cube, angles=angles, psf=psf, fwhm=fwhm_arcsec, pixscale=pixscale, verbose=False)

    gpu_sync = None
    try:
        import cupy as cp
        gpu_sync = cp.cuda.Stream.null.synchronize
    except Exception:
        pass

    if enable_gpu and gpu_sync is not None:
        # warmup
        _ = model.PACOCalc(phi0s[:32], use_subpixel_psf_astrometry=use_subpixel, cpu=1)
        gpu_sync()

    if gpu_sync is not None:
        gpu_sync()
    t0 = time.perf_counter()
    a, b = model.PACOCalc(phi0s, use_subpixel_psf_astrometry=use_subpixel, cpu=1)
    if gpu_sync is not None:
        gpu_sync()
    dt = time.perf_counter() - t0

    with np.errstate(divide="ignore", invalid="ignore"):
        snr = np.divide(b, np.sqrt(a), out=np.zeros_like(b), where=(a > 0))
        snr = np.nan_to_num(snr, nan=0.0, posinf=0.0, neginf=0.0)

    return {
        "time_s": float(dt),
        "a": np.asarray(a),
        "b": np.asarray(b),
        "snr": np.asarray(snr),
        "snr_max": float(np.nanmax(snr)),
        "snr_mean": float(np.nanmean(snr)),
    }


def compare_runs(ref, test, topk=20, thresholds=(3.0, 5.0, 7.0)):
    out = {}
    for key in ("a", "b", "snr"):
        x = np.asarray(ref[key])
        y = np.asarray(test[key])
        out[f"{key}_max_abs_diff"] = float(np.max(np.abs(x - y)))
        out[f"{key}_mean_abs_diff"] = float(np.mean(np.abs(x - y)))

    x = np.asarray(ref["snr"])
    y = np.asarray(test["snr"])
    out["snr_corr"] = float(np.corrcoef(x, y)[0, 1])

    idx_ref = np.argsort(-x)[:topk]
    idx_test = np.argsort(-y)[:topk]
    out["topk_overlap"] = float(len(set(idx_ref).intersection(set(idx_test))) / topk)

    for th in thresholds:
        out[f"count_ref_snr>{th}"] = int(np.sum(x > th))
        out[f"count_test_snr>{th}"] = int(np.sum(y > th))

    return out


In [ ]:
# Ejecutar comparacion robusta (mismo dataset, mismos phi0s)
cube, angles, psf, pixscale = load_dataset(DATASET_ROOT, INSTRUMENT, DATASET_ID)
cube = crop_cube_center(cube, half_size=CROP_HALF_SIZE)
phi0s = build_phi0_annulus(
    img_shape=(cube.shape[1], cube.shape[2]),
    n_pixels=N_PIXELS,
    inner=INNER_RADIUS,
    outer=OUTER_RADIUS,
)

print(f"Cube: {cube.shape}, phi0s: {len(phi0s)}")

cpu_res = run_paco(cube, angles, psf, pixscale, phi0s, enable_gpu=False, use_subpixel=USE_SUBPIXEL)
gpu_res_1 = run_paco(cube, angles, psf, pixscale, phi0s, enable_gpu=True, use_subpixel=USE_SUBPIXEL)
gpu_res_2 = run_paco(cube, angles, psf, pixscale, phi0s, enable_gpu=True, use_subpixel=USE_SUBPIXEL)

speedup = cpu_res["time_s"] / gpu_res_1["time_s"]
print(f"CPU: {cpu_res['time_s']:.2f}s | GPU: {gpu_res_1['time_s']:.2f}s | Speedup: {speedup:.2f}x")

cmp_cpu_gpu = compare_runs(cpu_res, gpu_res_1, topk=TOPK, thresholds=THRESHOLDS)
cmp_gpu_gpu = compare_runs(gpu_res_1, gpu_res_2, topk=TOPK, thresholds=THRESHOLDS)

df_summary = pd.DataFrame([
    {
        "run": "CPU",
        "time_s": cpu_res["time_s"],
        "snr_max": cpu_res["snr_max"],
        "snr_mean": cpu_res["snr_mean"],
    },
    {
        "run": "GPU_1",
        "time_s": gpu_res_1["time_s"],
        "snr_max": gpu_res_1["snr_max"],
        "snr_mean": gpu_res_1["snr_mean"],
    },
    {
        "run": "GPU_2",
        "time_s": gpu_res_2["time_s"],
        "snr_max": gpu_res_2["snr_max"],
        "snr_mean": gpu_res_2["snr_mean"],
    },
])

print("\nResumen de corridas:")
display(df_summary)

print("\nComparacion CPU vs GPU:")
display(pd.DataFrame([cmp_cpu_gpu]))

print("\nRepetibilidad GPU (GPU_1 vs GPU_2):")
display(pd.DataFrame([cmp_gpu_gpu]))


In [ ]:
# Criterios practicos de robustez (ajustables)
ok = {
    "snr_corr_cpu_gpu": cmp_cpu_gpu["snr_corr"] > 0.99,
    "topk_overlap_cpu_gpu": cmp_cpu_gpu["topk_overlap"] >= 0.8,
    "snr_corr_gpu_gpu": cmp_gpu_gpu["snr_corr"] > 0.999,
    "topk_overlap_gpu_gpu": cmp_gpu_gpu["topk_overlap"] >= 0.95,
}

print("Checks de robustez:")
for k, v in ok.items():
    print(f"- {k}: {'OK' if v else 'REVISAR'}")
